# SESAR legacy to SESAR2024 ETL

In [6]:
#import psycopg2
import requests
import pandas
import sqlalchemy




In [5]:
!dir ..\MigrationNotes_LookupTables



 Volume in drive C has no label.
 Volume Serial Number is 76FF-EC85

 Directory of C:\Users\smrTu\OneDrive\Documents\GithubC\IEDA\sesar2\MigrationNotes_LookupTables

09/03/2024  11:59 AM    <DIR>          .
09/03/2024  12:14 PM    <DIR>          ..
07/12/2024  01:35 PM             8,703 agentsMapping.md
08/16/2024  07:05 PM         5,240,082 agentsMapping.xlsx
08/21/2024  04:14 PM           174,940 AgentsTesting.png
08/21/2024  04:14 PM           128,187 GeologicTimeScale2020.txt
08/19/2024  10:48 AM            35,874 GeologicTimeScale2020.xlsx
09/02/2024  07:18 AM             8,539 MigrationWorkflowNotes.md
07/12/2024  01:51 PM            77,631 NewAgentsSchema.png
07/12/2024  01:51 PM           156,746 NewSample-AgentsSchema.png
08/21/2024  04:14 PM           264,519 PermissionsTesting.png
08/21/2024  04:14 PM           232,305 PermissionsTestingV2.png
08/19/2024  11:51 AM             9,921 PlatformsVocabWork.xlsx
08/21/2024  04:14 PM           301,067 Sample-Agents.png
08/21/2024  0

In [7]:
df = pandas.read_excel('../MigrationNotes_LookupTables/GeologicTimeScale2020.xlsx')
df.head

<bound method NDFrame.head of                                                    URI              label  \
0    http://resource.geosciml.org/classifier/ics/is...    Phanerozoic Eon   
1    http://resource.geosciml.org/classifier/ics/is...       Cenozoic Era   
2    http://resource.geosciml.org/classifier/ics/is...  Quaternary Period   
3    http://resource.geosciml.org/classifier/ics/is...     Holocene Epoch   
4    http://resource.geosciml.org/classifier/ics/is...     Meghalayan Age   
..                                                 ...                ...   
173  http://resource.geosciml.org/classifier/ics/is...     Neoarchean Era   
174  http://resource.geosciml.org/classifier/ics/is...    Mesoarchean Era   
175  http://resource.geosciml.org/classifier/ics/is...   Paleoarchean Era   
176  http://resource.geosciml.org/classifier/ics/is...      Eoarchean Era   
177  http://resource.geosciml.org/classifier/ics/is...         Hadean Eon   

         notation    beginMa  beginDateUncerM

In [ ]:
def get_legacyConnection():
    try:
        return psycopg2.connect(
            database="sesardb20240525",
            user="postgres",
            password="smrpostgis",
            host="127.0.0.1",
            port=5432,
        )
    except:
        return False


In [ ]:
def get_2024Connection():
    try:
        return psycopg2.connect(
            database="SESAR2024",
            user="postgres",
            password="smrpostgis",
            host="127.0.0.1",
            port=5432,
        )
    except:
        return False

In [ ]:
def executeQuery(conn, querystring):
# CREATE A CURSOR USING THE CONNECTION OBJECT
    curr = conn.cursor()
    # EXECUTE THE SQL QUERY
    curr.execute(querystring)
    # FETCH ALL THE ROWS FROM THE CURSOR
    data = curr.fetchall()
    return data

In [ ]:
def getFields(conn,tableName):
#  get field names for table
    fieldsquery="SELECT column_name,ordinal_position "\
     "FROM information_schema.columns " + \
     "WHERE table_schema = 'public' AND table_name = '" + \
        tableName + "' ORDER BY ordinal_position"
    print(fieldsquery)
    fields = executeQuery(conn, fieldsquery)

    fieldlist=[]
    for row in fields:
        fieldlist.append(row[0])
    return fieldlist

In [ ]:
def getTableData(conn, tableName, tablePK):
    fl=getFields(conn, tableName)
    print(tableName, " fields ", fl)
    selectRecordQuery = "SELECT * FROM public." + tableName 
    #print("record query: ", selectRecordQuery)

    data = executeQuery(conn, selectRecordQuery)
    # do something with THE RECORDS 
    print (tableName, " number of records: ", len(data))

    theDict={}
    rows=[]
    rowNum = 0
    for row in data:
        theobj={}
        arecord={}
        for nc in range(len(row)):
            if row[nc] is None:
                # don't put null values in JSON
                continue
            theobj[fl[nc]]=row[nc]
        #if tablePK=='noPK':
        #    arecord['id'+str(rowNum)]=theobj
        #else:
        #    arecord[theobj[tablePK]]=theobj
        #print(arecord)
        #rows.append(arecord)
        rows.append(theobj)
        rowNum = rowNum+1
    
    #print(rows)
    theDict[tableName] = rows
    return theDict

In [ ]:
def getPrimaryKey(conn, tableName):
    pkquery = "SELECT c.column_name " + \
        " FROM information_schema.key_column_usage AS c " + \
        " LEFT JOIN information_schema.table_constraints AS t " + \
        " ON t.constraint_name = c.constraint_name " + \
        " WHERE t.table_name = '" + tableName + "' AND t.constraint_type = 'PRIMARY KEY'; " 
    #print(pkquery)
    key = executeQuery(conn, pkquery)
    print("the primary key: ", key)
    if len(key) == 0:
        return "noPK"
    else:
        return key[0][0]

In [ ]:
# database connection is global variable.
oldDb = get_legacyConnection()
if oldDb:
    print("Connection to the Legacy PostgreSQL database established successfully.")
else:
    print("Connection to the Legacy PostgreSQL database encountered and error.")

In [ ]:
# database connection is global variable.
newDb = get_2024Connection()
if newDb:
    print("Connection to SESAR2024 PostgreSQL database established successfully.")
else:
    print("Connection to SESAR2024 PostgreSQL encountered and error.")

# set up to process a table

In [ ]:
# list of tables in database SELECT * FROM information_schema.tables WHERE table_schema = 'public' and table_type = 'BASE TABLE'
#  get field names for table
tablesquery="SELECT table_name " + \
 "FROM information_schema.tables " + \
 "WHERE table_schema = 'public' AND table_type = 'BASE TABLE'" 
print(tablesquery)
tables = executeQuery(newDb, tablesquery)

tablelist=[]
for row in tables:
    tablelist.append(row[0])
print(tablelist)


In [ ]:
def load_launch_type():
# table: launch_type.  Simple copy from olddb to newdb, field order is the same,
#  but new launch_type table has a source field, so have to append this in dictionary
#  with a default value 'missing'.
    tableName = 'launch_type'
    newTableName = 'launch_type'
    thepk = getPrimaryKey(oldDb, tableName)
    print('table: ', tableName, ' primaryKey: ', thepk)
    
    theDict = getTableData(oldDb, tableName, thepk)
    for row in theDict[tableName]:
        row['source'] = 'missing'
    
    #print(list(theDict[tableName][1].values()))
    #newDb.autocommit = True
    cursor = newDb.cursor()
    for anobj in theDict[tableName]:
        thevalues =  list(anobj.values())
        insertvalues = '('
        for val in thevalues:
            if isinstance(val, str):
                val = '\'' + val + '\''
            else:
                val = str(val)
            insertvalues = insertvalues + val + ', '
        
        print(insertvalues[:-2] + ')')
        sql2='''insert into ''' + newTableName + ''' (launch_type_id , label , 
              description , source) VALUES {};'''.format(insertvalues[:-2] + ')')
        print ('the query: ', sql2)
        cursor.execute(sql2)
    
    newDb.commit()
    return 1
    
    #sql3='''select * from ''' + newTableName + ''';'''
    #cursor.execute(sql3)
    #for i in cursor.fetchall():
    #    print(i)

In [ ]:
def load_country():
    # table: country.  Simple copy from olddb to newdb, new table has iso3166code field inserted
    # fields from existing table. 
    tableName = 'country'
    newTableName = 'country'
    thepk = getPrimaryKey(oldDb, tableName)
    print('table: ', tableName, ' primaryKey: ', thepk)
    
    theTempDict = getTableData(oldDb, tableName, thepk)
    theDict = {}
    rows=[]
    for row in theTempDict[tableName]:
        newRow = {}
        newRow['country_id'] = row['country_id']
        newRow['label'] = row['name']
        newRow['ido3166code'] = 'x'
        newRow['is_active'] = row['is_active']
        #print('newRow: ', newRow)
        rows.append(newRow)
    
    theDict[tableName] = rows
    #print(theDict)    
    
    #print(list(theDict[tableName][1].values()))
    #newDb.autocommit = True
    cursor = newDb.cursor()
    for anobj in theDict[tableName]:
        thevalues =  list(anobj.values())
        insertvalues = '('
        for val in thevalues:
            if isinstance(val, str):
                # Cote d'ivoire causes problem
                #escaping quote in insert query didn't work
                val= val.replace('\'','$')
                val = '\'' + val + '\''
            else:
                val = str(val)
            insertvalues = insertvalues + val + ', '
        
        print(insertvalues[:-2] + ')')
        sql2='''insert into ''' + newTableName + ''' (country_id , label , 
              iso3166code , is_active) VALUES {};'''.format(insertvalues[:-2] + ')')
        #print ('the query: ', sql2)
        cursor.execute(sql2)
    
    #newDb.commit()
    return 1
    
    #sql3='''select * from ''' + newTableName + ''';'''
    #cursor.execute(sql3)
    #for i in cursor.fetchall():
    #    print(i)


# Agent table
2024-06-20

 the agent table includes agents from sesar_user, sesar_user.institution, sesar_user.institution_detail,  sample.collector, and archive_lkup
  sesar_user.institution needs to be updated with peng_org.org_name, but not all
  sesar_user.institution have matching peng.org_name.  need new lookup to clean
  sesar_user.institution, with the peng mappings, and new mapping for 
  stuff peng didn't map.
  
 query for institution lookup table csv export:
 
```
SELECT distinct sesar.institution, sesar.institution_detail, peng.org_name 
	FROM public.sesar_user as sesar left join public.peng_org as peng 
	on peng.sesar_name = sesar.institution order by sesar.institution
```

lookup from archive_lkup to peng.org to see what matches:
only get a couple matches
Also have archive_mapping table that normalizes archive names-- use this for agents. 
archive_lkup is not sync'd with what is in the db, so construct query to get archive names from sample.current_archive and sample.original_archive

query:

```
SELECT map.archive_mapping_id, 
		sam.current_archive as archive_org, 
		sam.current_archive_contact as archive_contact, 
		map.preferred_archive_name, 
		peng.org_name as peng_name
	FROM  public.sample as sam 
		  LEFT JOIN public.archive_mapping as map
			on map.user_entered_archive_name = sam.current_archive
		  LEFT JOIN public.peng_org as peng 
			on peng.sesar_name = sam.current_archive
UNION 
SELECT	map.archive_mapping_id, 
		sam2.original_archive as archive_org, 
		sam2.original_archive_contact as archive_contact, 
		map.preferred_archive_name, 
		peng.org_name as peng_name
	FROM  public.sample as sam2 
		 LEFT JOIN public.archive_mapping as map
			on map.user_entered_archive_name = sam2.original_archive
		 LEFT JOIN public.peng_org as peng 
			on peng.sesar_name = sam2.original_archive 
ORDER BY archive_org ASC 
```

need to concatenate these to get organizations in 2024 agents table.  
PK agent_id will start with copying sesar_user.sesar_user_id.  
agent_id from the merged organizations will start with max(sesar_user.sesar_user_id) and increment from there. 

for people, all orcids in ecl_orcid_temp are in sesar_user.orcid, so the ecl_orcid_temp table is not needed.

need to append collectors to agents draft compilation and harmonize with sesar_users. Collector_lkup is out of sync with public.sample.collector; need to use collectors in authority table, collector_lkup does not appear useful.  Query:

```
SELECT distinct  sam.collector as sam_collector, lkup.name as lkup_name,
	s.fname, s.lname
	FROM public.sample as sam left join public.collector_lkup as lkup
	on sam.collector = lkup.name
	left join sesar_user as s  on sam.collector = s.orcid
ORDER BY sam.collector ASC 
```

Note that most of the orcids in the collector.lkup do not match orcid's in sesar_user.

2024-08-16
lots of work done to generate update mappings for individuals and organizations.  Correlating names between the various SESAR tables with free text names.  Big challenge with 


In [ ]:
#def load_xxxx():
# table: Agent. . 
tableName = 'sesar_user'
newTableName = 'agent'
thepk = getPrimaryKey(oldDb, tableName)
print('table: ', tableName, ' primaryKey: ', thepk)

theTempDict = getTableData(oldDb, tableName, thepk)
theDict = {}
rows=[]
for row in theTempDict[tableName]:
    newRow = {}
    newRow['agent_id'] = row['sesar_user_id']
    newRow['label'] = row['']
    newRow['fname'] = row['fname']
    newRow['lname'] = row['']
    newRow['agent_uri'] = row['orcid']
    newRow['address'] = row['address1'] + ', ' + row['address2'] + ', ' + row['city'] + ', ' + \
            row['state_province'] + ', ' + row['postal_code'] 
    newRow['agent_type'] = 'individual'
    poc=''
    if not is None row['email']: 
        poc = 'email: ' + row['email']
    if not is None row['phone']:
        if len(poc)==0 :
            poc = 'phone: ' + row['phone']
        else:
            poc = poc + '; phone: ' + row['phone']
    newRow['point_of_contact'] = poc
    newRow['organization_affiliation_id'] = row['']
    newRow['parent_organization_id'] = row['']
    #print('newRow: ', newRow)
    rows.append(newRow)

theDict[tableName] = rows
#print(theDict)    



In [ ]:
# load the geologic_time_scale table


In [ ]:
#def load_xxxx():
# table: Sample. 
tableName = 'sample'
newTableName = 'sample'

thepk = getPrimaryKey(oldDb, tableName)
print('table: ', tableName, ' primaryKey: ', thepk)

theTempDict = getTableData(oldDb, tableName, thepk)
theDict = {}
rows=[]
for row in theTempDict[tableName]:
    #start by populating fields that copy directly
    newRow = {}
    newRow['agent_id'] = row['sesar_user_id']


    

In [ ]:
    # CLOSE THE CONNECTION
newDb.close()
oldDb.close()